In [3]:
import os

# Xóa HADOOP_HOME khỏi session này để chạy local
os.environ.pop("HADOOP_HOME", None)
os.environ.pop("HADOOP_CONF_DIR", None)

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Lab4").master("local[*]").getOrCreate()
print(spark.version)

3.5.0


In [34]:
from pyspark.sql.functions import (
    col, count, countDistinct, avg, sum as spark_sum,
    round as spark_round, dense_rank, datediff,
    max, min, when, year, month
)
from pyspark.sql.window import Window
import builtins

**1.	Hãy đọc dữ liệu từ các file csv, sử dụng tự suy ra kiểu dữ liệu cho mỗi cột.**

In [35]:
# Đọc tất cả file, tự suy luận schema (inferSchema=True)
orders = spark.read.csv("Data/Orders.csv", header=True, inferSchema=True, sep=";")
customers = spark.read.csv("Data/Customer_List.csv", header=True, inferSchema=True, sep=";")
order_items = spark.read.csv("Data/Order_Items.csv", header=True, inferSchema=True, sep=";")
products = spark.read.csv("Data/Products.csv", header=True, inferSchema=True, sep=";")
reviews = spark.read.csv("Data/Order_Reviews.csv", header=True, inferSchema=True, sep=";")

In [40]:
def print_schema(df, name, file=None):
    lines = []
    lines.append(f"{'='*50}")
    lines.append(f"  {name} ({df.count()} rows)")
    lines.append(f"{'='*50}")
    for field in df.schema.fields:
        nullable = "nullable" if field.nullable else "not null"
        lines.append(f"  {field.name:<40} {str(field.dataType):<20} ({nullable})")
    lines.append("")
    
    output = "\n".join(lines)
    print(output)
    if file:
        file.write(output + "\n")

with open("bai1.txt", "w", encoding="utf-8") as f:
    print_schema(orders, "Orders", f)
    print_schema(customers, "Customer_List", f)
    print_schema(order_items, "Order_Items", f)
    print_schema(products, "Products", f)
    print_schema(reviews, "Order_Reviews", f)

print("Đã lưu vào bai1.txt")

  Orders (99441 rows)
  Order_ID                                 StringType()         (nullable)
  Customer_Trx_ID                          StringType()         (nullable)
  Order_Status                             StringType()         (nullable)
  Order_Purchase_Timestamp                 TimestampType()      (nullable)
  Order_Approved_At                        TimestampType()      (nullable)
  Order_Delivered_Carrier_Date             TimestampType()      (nullable)
  Order_Delivered_Customer_Date            TimestampType()      (nullable)
  Order_Estimated_Delivery_Date            TimestampType()      (nullable)

  Customer_List (102727 rows)
  Customer_Trx_ID                          StringType()         (nullable)
  Subscriber_ID                            StringType()         (nullable)
  Subscribe_Date                           DateType()           (nullable)
  First_Order_Date                         DateType()           (nullable)
  Customer_Postal_Code                     Stri

**2.	Thống kê tổng số đơn hàng, số lượng khách hàng và người bán.**

In [41]:
total_orders = orders.count()
total_customers = customers.select(countDistinct("Subscriber_ID")).collect()[0][0]
total_trx_customers = orders.select(countDistinct("Customer_Trx_ID")).collect()[0][0]
total_sellers = order_items.select(countDistinct("Seller_ID")).collect()[0][0]

lines = [
    "=" * 60,
    f"  {'Thống kê tổng quan':^60}",
    "=" * 60,
    f"  {'Tổng số đơn hàng':<50} {total_orders:>6,}",
    f"  {'Số lượng khách hàng thật sự':<50} {total_customers:>6,}",
    f"  {'Số lượng khách hàng đã thực hiện giao dịch':<50} {total_trx_customers:>6,}",
    f"  {'Số lượng người bán':<50} {total_sellers:>6,}",
    "=" * 60,
]

output = "\n".join(lines)
print(output)

with open("bai2.txt", "w", encoding="utf-8") as f:
    f.write(output + "\n")

print("Đã lưu vào bai2.txt")

                       Thống kê tổng quan                     
  Tổng số đơn hàng                                   99,441
  Số lượng khách hàng thật sự                        99,382
  Số lượng khách hàng đã thực hiện giao dịch         99,441
  Số lượng người bán                                  3,095
Đã lưu vào bai2.txt


**3.	Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần.**

In [42]:
orders_by_country = orders.join(customers, on="Customer_Trx_ID", how="left") \
    .groupBy("Customer_Country").agg(count("Order_ID").alias("Số_đơn_hàng")) \
    .orderBy("Số_đơn_hàng", ascending=False)

orders_by_country.show(truncate=False)

# Lưu kết quả
result = orders_by_country.collect()
with open("bai3.txt", "w", encoding="utf-8") as f:
    f.write(f"{'Quốc gia':<30} {'Số đơn hàng':>12}\n")
    f.write("-" * 44 + "\n")
    for row in result:
        country = str(row["Customer_Country"]) if row["Customer_Country"] else "NULL"
        f.write(f"{country:<30} {row['Số_đơn_hàng']:>12,}\n")

print(f"\nĐã lưu {len(result)} dòng vào bai3.txt")

+----------------+-----------+
|Customer_Country|Số_đơn_hàng|
+----------------+-----------+
|Germany         |41754      |
|France          |12848      |
|Netherlands     |11629      |
|Belgium         |5464       |
|Austria         |5043       |
|Switzerland     |3640       |
|United Kingdom  |3382       |
|Poland          |2139       |
|Czechia         |2034       |
|Italy           |2025       |
|Spain           |1651       |
|Portugal        |1336       |
|Sweden          |975        |
|Denmark         |905        |
|Serbia          |746        |
|Norway          |716        |
|Slovakia        |534        |
|Slovenia        |495        |
|Turkey          |485        |
|Greece          |412        |
+----------------+-----------+
only showing top 20 rows


Đã lưu 27 dòng vào bai3.txt


**4.	Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng (Hiển thị theo năm tăng dần, tháng giảm dần)**

In [43]:
orders_by_month = orders.groupBy(
        year("Order_Purchase_Timestamp").alias("Năm"),
        month("Order_Purchase_Timestamp").alias("Tháng")
    ).agg(count("Order_ID").alias("Số_đơn_hàng")) \
    .orderBy("Năm", ascending=True) \
    .orderBy("Năm", "Tháng", ascending=[True, False])

orders_by_month.show(36, truncate=False)

# Lưu vào file
result = orders_by_month.collect()

with open("bai4.txt", "w", encoding="utf-8") as f:
    f.write(f"{'Năm':<8} {'Tháng':<8} {'Số đơn hàng':>12}\n")
    f.write("-" * 30 + "\n")
    for row in result:
        f.write(f"{row['Năm']:<8} {row['Tháng']:<8} {row['Số_đơn_hàng']:>12,}\n")

print(f"Đã lưu {len(result)} dòng vào bai4.txt")

+----+-----+-----------+
|Năm |Tháng|Số_đơn_hàng|
+----+-----+-----------+
|2022|12   |1          |
|2022|10   |324        |
|2022|9    |4          |
|2023|12   |5673       |
|2023|11   |7544       |
|2023|10   |4631       |
|2023|9    |4285       |
|2023|8    |4331       |
|2023|7    |4026       |
|2023|6    |3245       |
|2023|5    |3700       |
|2023|4    |2404       |
|2023|3    |2682       |
|2023|2    |1780       |
|2023|1    |800        |
|2024|10   |4          |
|2024|9    |16         |
|2024|8    |6512       |
|2024|7    |6292       |
|2024|6    |6167       |
|2024|5    |6873       |
|2024|4    |6939       |
|2024|3    |7211       |
|2024|2    |6728       |
|2024|1    |7269       |
+----+-----+-----------+

Đã lưu 25 dòng vào bai4.txt


**5.	Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức (ví dụ: 1 đến 5).  
Lưu ý: Cần xử lý các giá trị ngoại lệ và NULL trong cột Review_Score**

In [44]:
# Quan sát phân phối giá trị của review_score
reviews.groupBy("Review_Score").count().orderBy("Review_Score").show(20)

+----------------+-----+
|    Review_Score|count|
+----------------+-----+
|            NULL|   45|
|               1|11424|
|               2| 3151|
|2024-04-07 04:19|    1|
|2024-07-05 11:11|    1|
|               3| 8179|
|               4|19141|
|               5|57328|
+----------------+-----+



In [45]:
# Bỏ qua các ngoại lệ là NULL, datetime để tính review score
# Lọc chỉ giữ giá trị hợp lệ 1-5
reviews_clean = reviews.filter(
    col("Review_Score").isin("1", "2", "3", "4", "5")
)

# Tính điểm trung bình và số lượng theo từng mức
review_stats = reviews_clean.groupBy("Review_Score").agg(count("Review_ID").alias("Số_đánh_giá")).orderBy("Review_Score")
avg_score = reviews_clean.selectExpr("avg(cast(Review_Score as double)) as avg_score").collect()[0]["avg_score"]

# Hiển thị
review_stats.show(truncate=False)
print(f"Điểm đánh giá trung bình: {avg_score:.2f}")
print(f"Số dòng bị loại (NULL + ngoại lệ): {reviews.count() - reviews_clean.count()}")

# Lưu file
result = review_stats.collect()
with open("bai5.txt", "w", encoding="utf-8") as f:
    f.write(f"{'Mức đánh giá':<15} {'Số đánh giá':>12}\n")
    f.write("-" * 29 + "\n")
    for row in result:
        f.write(f"{row['Review_Score']:<15} {row['Số_đánh_giá']:>12,}\n")
    f.write("-" * 29 + "\n")
    f.write(f"{'Điểm TB':<15} {avg_score:>12.2f}\n")
    f.write(f"{'Dòng bị loại':<15} {reviews.count() - reviews_clean.count():>12,}\n")

print("Đã lưu vào bai5.txt")

+------------+-----------+
|Review_Score|Số_đánh_giá|
+------------+-----------+
|1           |11424      |
|2           |3151       |
|3           |8179       |
|4           |19141      |
|5           |57328      |
+------------+-----------+

Điểm đánh giá trung bình: 4.09
Số dòng bị loại (NULL + ngoại lệ): 47
Đã lưu vào bai5.txt


**6.  Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo danh mục sản phẩm**

In [46]:
revenue_2024 = order_items.join(orders, on="Order_ID", how="inner") \
    .join(products, on="Product_ID", how="left") \
    .filter(year("Order_Purchase_Timestamp") == 2024) \
    .groupBy("Product_Category_Name") \
    .agg(
        round(sum(col("Price") + col("Freight_Value")), 2).alias("Doanh_thu"),
        sum(col("Price")).alias("Tiền_hàng"),
        sum(col("Freight_Value")).alias("Phí_vận_chuyển")
    ) \
    .orderBy("Doanh_thu", ascending=False)

revenue_2024.show(100, truncate=False)

# Lưu file
result = revenue_2024.collect()
total = builtins.sum(row["Doanh_thu"] for row in result)

with open("bai6.txt", "w", encoding="utf-8") as f:
    f.write(f"{'Danh mục':<35} {'Doanh thu':>15} {'Tiền hàng':>15} {'Phí vận chuyển':>15}\n")
    f.write("-" * 82 + "\n")
    for row in result:
        cat = str(row["Product_Category_Name"]) if row["Product_Category_Name"] else "NULL"
        f.write(f"{cat:<35} {row['Doanh_thu']:>15,.2f} {row['Tiền_hàng']:>15,.2f} {row['Phí_vận_chuyển']:>15,.2f}\n")
    f.write("-" * 82 + "\n")
    f.write(f"{'TỔNG':<35} {total:>15,.2f}\n")

print(f"Đã lưu {len(result)} danh mục vào bai6.txt")

+---------------------------------------+---------+------------------+------------------+
|Product_Category_Name                  |Doanh_thu|Tiền_hàng         |Phí_vận_chuyển    |
+---------------------------------------+---------+------------------+------------------+
|Health_Beauty                          |885191.12|772238.1499999929 |112952.96999999981|
|Watches_Gifts                          |771986.75|708850.9399999969 |63135.81000000006 |
|Bed_Bath_Table                         |650794.7 |538069.2599999947 |112725.44000000003|
|Sports_Leisure                         |621999.34|532566.4899999963 |89432.84999999995 |
|Computers_Accessories                  |594771.04|505476.3099999975 |89294.72999999997 |
|Housewares                             |491576.96|399888.0999999989 |91688.85999999984 |
|Furniture_Decor                        |476466.13|386668.58999999857|89797.54          |
|Auto                                   |404210.57|347631.1499999995 |56579.42          |
|Baby     

**7.  Xác định sản phẩm có số lượng bán ra cao nhất và tính điểm đánh giá trung bình cho từng sản phẩm**

In [47]:
# Làm sạch Review_Score trước
reviews_clean = reviews.filter(col("Review_Score").isin("1","2","3","4","5")) \
    .withColumn("Review_Score", col("Review_Score").cast("double"))

# Join và tính toán
product_stats = order_items \
    .join(reviews_clean, on="Order_ID", how="left") \
    .join(products, on="Product_ID", how="left") \
    .groupBy("Product_ID", "Product_Category_Name") \
    .agg(
        count("Order_Item_ID").alias("Số_lượng_bán"),
        spark_round(avg("Review_Score"), 4).alias("Điểm_TB")
    ) \
    .orderBy("Số_lượng_bán", ascending=False)

product_stats.show(20, truncate=False)

# Lưu file
result = product_stats.collect()
with open("bai7.txt", "w", encoding="utf-8") as f:
    f.write(f"{'Product_ID':<35} {'Danh mục':<30} {'Số lượng bán':>14} {'Điểm TB':>10}\n")
    f.write("-" * 92 + "\n")
    for row in result:
        cat = str(row["Product_Category_Name"]) if row["Product_Category_Name"] else "NULL"
        score = f"{row['Điểm_TB']:.4f}" if row["Điểm_TB"] else "NULL"
        f.write(f"{row['Product_ID']:<35} {cat:<30} {row['Số_lượng_bán']:>14,} {score:>10}\n")

print(f"Đã lưu {len(result)} sản phẩm vào bai7.txt")

+--------------------------------+---------------------+------------+-------+
|Product_ID                      |Product_Category_Name|Số_lượng_bán|Điểm_TB|
+--------------------------------+---------------------+------------+-------+
|aca2eb7d00ea1a7b8ebd4e68314663af|Furniture_Decor      |527         |4.0191 |
|99a4788cb24856965c36a24e339b6058|Bed_Bath_Table       |491         |3.8983 |
|422879e10f46682990de24d770e7f83d|Garden_Tools         |487         |3.9465 |
|389d119b48cf3043d311335e499d9c6b|Garden_Tools         |392         |4.1176 |
|368c6c730842d78016ad823897a372db|Garden_Tools         |391         |3.9227 |
|53759a2ecddad2bb87a079a1f1519f73|Garden_Tools         |375         |3.8686 |
|d1c427060a0f73f6b889a5c7c61f2ac4|Computers_Accessories|343         |4.1941 |
|53b36df67ebb7c41585e8d54d6772e08|Watches_Gifts        |323         |4.1906 |
|154e7e31ebfa092203795c972e5804a6|Health_Beauty        |293         |4.3151 |
|3dd2a17168ec895c781a9191c1e95ad7|Computers_Accessories|274     

**8.  Tính toán hiệu số giữa ngày giao hàng thực tế (Order_Delivered_Carrier_Date) và ngày giao hàng dự kiến (ví dụ: Shipping_Limit_Date từ bảng Order_Items) để đánh giá hiệu suất giao hàng.**

In [48]:
# So sánh giữa Order_Delivered_Customer_Date với Order_Estimated_Delivery_Date
# Tính hiệu số: thực tế - dự kiến
# Dương = trễ, Âm = sớm
delivery_perf = orders \
    .filter(col("Order_Delivered_Customer_Date").isNotNull()) \
    .withColumn(
        "Hiệu_số_ngày",
        datediff(col("Order_Delivered_Customer_Date"), col("Order_Estimated_Delivery_Date"))
    ).withColumn(
        "Trạng_thái",
        when(col("Hiệu_số_ngày") <= 0, "Đúng/Sớm hạn")
        .otherwise("Trễ hạn")
    )

# Thống kê tổng quan
summary = delivery_perf.agg(
    count("Order_ID").alias("Tổng_đơn"),
    spark_round(avg("Hiệu_số_ngày"), 2).alias("Hiệu_số_TB"),
    count(when(col("Trạng_thái") == "Đúng/Sớm hạn", 1)).alias("Đúng_Sớm_hạn"),
    count(when(col("Trạng_thái") == "Trễ hạn", 1)).alias("Trễ_hạn")
)

summary.show(truncate=False)

# Thống kê theo trạng thái
delivery_perf.groupBy("Trạng_thái") \
    .agg(
        count("Order_ID").alias("Số_đơn"),
        spark_round(avg("Hiệu_số_ngày"), 2).alias("Hiệu_số_TB")
    ) \
    .orderBy("Trạng_thái") \
    .show(truncate=False)

# Lưu file
result_summary = summary.collect()
result_detail = delivery_perf.groupBy("Trạng_thái") \
    .agg(
        count("Order_ID").alias("Số_đơn"),
        spark_round(avg("Hiệu_số_ngày"), 2).alias("Hiệu_số_TB")
    ).orderBy("Trạng_thái").collect()

with open("bai8.txt", "w", encoding="utf-8") as f:
    f.write("=== THỐNG KÊ HIỆU SUẤT GIAO HÀNG ===\n\n")
    f.write("--- Tổng quan ---\n")
    f.write(f"Tổng đơn hàng:       {result_summary[0]['Tổng_đơn']:>10,}\n")
    f.write(f"Hiệu số ngày TB:     {result_summary[0]['Hiệu_số_TB']:>10.2f}\n")
    f.write(f"Đúng/Sớm hạn:        {result_summary[0]['Đúng_Sớm_hạn']:>10,}\n")
    f.write(f"Trễ hạn:             {result_summary[0]['Trễ_hạn']:>10,}\n\n")
    f.write("--- Theo trạng thái ---\n")
    f.write(f"{'Trạng thái':<20} {'Số đơn':>10} {'Hiệu số TB (ngày)':>18}\n")
    f.write("-" * 50 + "\n")
    for row in result_detail:
        f.write(f"{row['Trạng_thái']:<20} {row['Số_đơn']:>10,} {row['Hiệu_số_TB']:>18.2f}\n")

print("Đã lưu vào bai8.txt")

+--------+----------+------------+-------+
|Tổng_đơn|Hiệu_số_TB|Đúng_Sớm_hạn|Trễ_hạn|
+--------+----------+------------+-------+
|96476   |-11.91    |89941       |6535   |
+--------+----------+------------+-------+

+------------+------+----------+
|Trạng_thái  |Số_đơn|Hiệu_số_TB|
+------------+------+----------+
|Trễ hạn     |6535  |10.64     |
|Đúng/Sớm hạn|89941 |-13.55    |
+------------+------+----------+

Đã lưu vào bai8.txt


**9.  Nhóm khách hàng dựa trên số lượng đơn hàng, giá trị trung bình của đơn hàng và tần suất mua sắm.**

In [51]:
# Tính giá trị mỗi đơn hàng từ order_items
order_value = order_items.groupBy("Order_ID") \
    .agg(spark_round(avg(col("Price") + col("Freight_Value")), 2).alias("Giá_trị_đơn"))

# Tính chỉ số cho từng khách hàng
customer_stats = orders \
    .join(customers, on="Customer_Trx_ID", how="inner") \
    .join(order_value, on="Order_ID", how="left") \
    .groupBy("Subscriber_ID") \
    .agg(
        count("Order_ID").alias("Số_đơn_hàng"),
        spark_round(avg("Giá_trị_đơn"), 2).alias("Giá_trị_TB"),
        spark_round(
            datediff(max("Order_Purchase_Timestamp"), min("Order_Purchase_Timestamp"))
            / count("Order_ID"), 2
        ).alias("Tần_suất_ngày")  # TB số ngày giữa các đơn
    )

# Phân nhóm khách hàng
customer_groups = customer_stats \
    .withColumn(
        "Nhóm",
        when(col("Số_đơn_hàng") == 1, "One-time")
        .when(col("Số_đơn_hàng") <= 3, "Occasional")
        .otherwise("Loyal")
    )

# Thống kê theo nhóm
customer_groups.groupBy("Nhóm") \
    .agg(
        count("Subscriber_ID").alias("Số_khách"),
        spark_round(avg("Số_đơn_hàng"), 2).alias("Đơn_TB"),
        spark_round(avg("Giá_trị_TB"), 2).alias("Giá_trị_TB"),
        spark_round(avg("Tần_suất_ngày"), 2).alias("Tần_suất_TB_ngày")
    ) \
    .orderBy("Số_khách", ascending=False) \
    .show(truncate=False)

# Lưu file
result_groups = customer_groups.groupBy("Nhóm") \
    .agg(
        count("Subscriber_ID").alias("Số_khách"),
        spark_round(avg("Số_đơn_hàng"), 2).alias("Đơn_TB"),
        spark_round(avg("Giá_trị_TB"), 2).alias("Giá_trị_TB"),
        spark_round(avg("Tần_suất_ngày"), 2).alias("Tần_suất_TB_ngày")
    ) \
    .orderBy("Số_khách", ascending=False) \
    .collect()

with open("bai9.txt", "w", encoding="utf-8") as f:
    f.write("=== PHÂN NHÓM KHÁCH HÀNG ===\n\n")
    f.write("Tiêu chí phân nhóm:\n")
    f.write("  One-time   : 1 đơn hàng\n")
    f.write("  Occasional : 2-3 đơn hàng\n")
    f.write("  Loyal      : >= 4 đơn hàng\n\n")
    f.write(f"{'Nhóm':<15} {'Số khách':>10} {'Đơn TB':>8} {'Giá trị TB':>12} {'Tần suất (ngày)':>16}\n")
    f.write("-" * 65 + "\n")
    for row in result_groups:
        f.write(f"{row['Nhóm']:<15} {row['Số_khách']:>10,} {row['Đơn_TB']:>8.2f} "
                f"{row['Giá_trị_TB']:>12.2f} {row['Tần_suất_TB_ngày']:>16.2f}\n")

print("Đã lưu vào bai9.txt")

+----------+--------+------+----------+----------------+
|Nhóm      |Số_khách|Đơn_TB|Giá_trị_TB|Tần_suất_TB_ngày|
+----------+--------+------+----------+----------------+
|One-time  |93099   |1.0   |147.35    |0.0             |
|Occasional|2948    |2.07  |129.17    |41.14           |
|Loyal     |49      |4.96  |131.59    |44.42           |
+----------+--------+------+----------+----------------+

Đã lưu vào bai9.txt


**10. Xếp hạng các seller dựa trên tổng doanh thu và số lượng đơn hàng bán được.**

In [52]:
seller_stats = order_items \
    .join(orders, on="Order_ID", how="inner") \
    .groupBy("Seller_ID") \
    .agg(
        spark_round(spark_sum(col("Price") + col("Freight_Value")), 2).alias("Tổng_doanh_thu"),
        countDistinct("Order_ID").alias("Số_đơn_hàng"),
        spark_round(spark_sum("Price"), 2).alias("Tiền_hàng"),
        spark_round(spark_sum("Freight_Value"), 2).alias("Phí_vận_chuyển")
    )

# Xếp hạng theo doanh thu
window_spec = Window.orderBy(col("Tổng_doanh_thu").desc())

seller_ranked = seller_stats \
    .withColumn("Xếp_hạng", dense_rank().over(window_spec)) \
    .orderBy("Xếp_hạng")

seller_ranked.show(20, truncate=False)

# Lưu file
result = seller_ranked.collect()

with open("bai10.txt", "w", encoding="utf-8") as f:
    f.write("=== XẾP HẠNG SELLER ===\n\n")
    f.write(f"{'Hạng':>6} {'Seller_ID':<35} {'Tổng DT':>15} {'Số đơn':>8} {'Tiền hàng':>15} {'Phí VC':>12}\n")
    f.write("-" * 95 + "\n")
    for row in result:
        f.write(f"{row['Xếp_hạng']:>6} {row['Seller_ID']:<35} "
                f"{row['Tổng_doanh_thu']:>15,.2f} {row['Số_đơn_hàng']:>8,} "
                f"{row['Tiền_hàng']:>15,.2f} {row['Phí_vận_chuyển']:>12,.2f}\n")

print(f"Đã lưu {len(result)} seller vào bai10.txt")

+--------------------------------+--------------+-----------+---------+--------------+--------+
|Seller_ID                       |Tổng_doanh_thu|Số_đơn_hàng|Tiền_hàng|Phí_vận_chuyển|Xếp_hạng|
+--------------------------------+--------------+-----------+---------+--------------+--------+
|4869f7a5dfa277a7dca6462dcf3b52b2|249640.7      |1132       |229472.63|20168.07      |1       |
|7c67e1448b00f6e969d365cea6b010ab|239536.44     |982        |187923.89|51612.55      |2       |
|53243585a1d6dc2643021fd1853d8905|235856.68     |358        |222776.05|13080.63      |3       |
|4a3ca9315b744ce9f8e9374361493884|235539.96     |1806       |200472.92|35067.04      |4       |
|fa1c13f2614d7b5c4749cbc52fecda94|204084.73     |585        |194042.03|10042.7       |5       |
|da8622b14eb17ae2831f4ac5b9dab84a|185192.32     |1314       |160236.57|24955.75      |6       |
|7e93a43ef30c4f03f38b393420bc753a|182754.05     |336        |176431.87|6322.18       |7       |
|1025f0e2d44d7041d6cf58b6550e0bfa|172860